In [ ]:
# Celda 1: imports, seeds y carga del dataset final
# Este notebook arranca desde el dataset ya armado por
# cnn_damage_classification_ETL_final.ipynb (hay que correrlo antes).
# Espera dos archivos guardados por ese notebook:
#   - data_clasificacion/dataset_final.csv  -> columnas image_path, label
#     (una sola superclase por imagen, ya sin duplicados ni corruptas)
#   - data_clasificacion/grupos_final.json  -> el diccionario GRUPOS_FINAL
# MODO_PRUEBA = True entrena con solo el 10% de las imagenes (misma
# proporcion por clase) para probar rapido que todo corre de punta a punta;
# para el entrenamiento real dejarlo en False.
# Fija seeds para reproducibilidad y define FINAL_CLASSES (orden fijo
# de las clases) y el mapeo nombre de clase <-> numero.

import os
import json
import random

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from PIL import Image

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

LOCAL_DATA_DIR = "data_clasificacion"
MODO_PRUEBA = False

df = pd.read_csv(f"{LOCAL_DATA_DIR}/dataset_final.csv")
with open(f"{LOCAL_DATA_DIR}/grupos_final.json") as f:
    GRUPOS_FINAL = json.load(f)

FINAL_CLASSES = list(GRUPOS_FINAL)
clase_a_idx = {cls: i for i, cls in enumerate(FINAL_CLASSES)}
df["label_idx"] = df["label"].map(clase_a_idx)

if MODO_PRUEBA:
    df = df.groupby("label", group_keys=False).sample(frac=0.10, random_state=42).reset_index(drop=True)
    print("MODO_PRUEBA activado: se usa el 10% del dataset")

print(f"Imagenes: {len(df)} | Clases: {len(FINAL_CLASSES)}")
print(df["label"].value_counts())

In [ ]:
# Celda 2: split train/val/test (70/15/15)
# Se separa en dos pasos con train_test_split estratificado por label:
# primero 70% train / 30% resto, y el resto se parte a la mitad en
# val y test. Asi cada split tiene la misma proporcion de cada clase.
# El test no se toca hasta la celda 10.
# Depende de: df, FINAL_CLASSES (celda 1)

train_df, resto_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=42)
val_df, test_df = train_test_split(resto_df, test_size=0.50, stratify=resto_df["label"], random_state=42)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"train: {len(train_df)} | val: {len(val_df)} | test: {len(test_df)}\n")
print(pd.DataFrame({
    "train": train_df["label"].value_counts(),
    "val": val_df["label"].value_counts(),
    "test": test_df["label"].value_counts(),
}).loc[FINAL_CLASSES])

In [ ]:
# Celda 3: desbalance con class weights
# Calcula un peso por clase inverso a su frecuencia en train
# (modo "balanced" de sklearn): las clases con menos imagenes pesan mas
# en la loss. Solo se usa en model.fit (entrenamiento), no afecta
# a validacion ni a test.
# Depende de: train_df, FINAL_CLASSES (celdas 1 y 2)

pesos = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(FINAL_CLASSES)),
    y=train_df["label_idx"].values,
)
class_weights = {i: float(p) for i, p in enumerate(pesos)}

for i, cls in enumerate(FINAL_CLASSES):
    print(f"  {cls:30s} peso {class_weights[i]:.2f}")

In [ ]:
# Celda 4: pipeline de imagenes con tf.data
# Lee cada imagen, la redimensiona a 224x224 y arma batches.
# En train mezcla y aplica augmentation (flip, rotacion leve, zoom
# y brillo); en val y test NO se mezcla ni se aumenta, asi el orden
# de las predicciones coincide con el del dataframe.
# Las etiquetas van como numero (label_idx) porque usamos
# sparse_categorical_crossentropy.
# Depende de: train_df, val_df, test_df (celda 2)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    return img, label

data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.1),
])

def make_dataset(dataframe, training):
    paths = dataframe["image_path"].values
    labels = dataframe["label_idx"].values.astype("int32")
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(2000, seed=42)
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)

In [ ]:
# Celda 5: modelo
# EfficientNetB0 preentrenada en ImageNet y congelada, con una cabeza
# nueva: pooling, dropout, dense y una capa final softmax con una salida
# por superclase (multi-clase: las probabilidades suman 1 y la
# prediccion es la clase con mayor probabilidad).
# Loss sparse_categorical_crossentropy porque las etiquetas son numeros.
# Depende de: IMG_SIZE (celda 4), FINAL_CLASSES (celda 1)

base_model = EfficientNetB0(include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,))
base_model.trainable = False

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = tf.keras.applications.efficientnet.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(FINAL_CLASSES), activation="softmax")(x)

model = models.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

In [ ]:
# Celda 6: entrenamiento fase 1 (solo la cabeza nueva)
# La base EfficientNet sigue congelada, solo se entrenan las capas nuevas
# con learning rate 1e-3. EarlyStopping corta si val_loss no mejora en
# 5 epocas y vuelve a los mejores pesos; ReduceLROnPlateau baja el LR
# si se estanca; ModelCheckpoint guarda el mejor modelo en
# ../models/modelo_clasificacion2.keras.
# Usa los class_weights de la celda 3.
# Depende de: model, train_ds, val_ds, class_weights

MODEL_PATH = "../models/modelo_clasificacion2.keras"

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3),
    ModelCheckpoint(MODEL_PATH, monitor="val_loss", save_best_only=True),
]

history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks,
)

In [ ]:
# Celda 7: entrenamiento fase 2 (fine-tuning)
# Descongela el ultimo 20% de las capas de EfficientNet y sigue
# entrenando con learning rate bajo (1e-5) para que la red se ajuste a
# las imagenes de danos sin romper lo que aprendio en ImageNet.
# Las capas BatchNormalization se dejan congeladas (recomendado para
# fine-tuning de EfficientNet). Hay que recompilar despues de cambiar
# que capas son entrenables.
# Despues carga el mejor modelo guardado por el checkpoint (menor val_loss
# entre las dos fases), asi la evaluacion y el guardado usan ese y no los
# pesos de la ultima epoca. Al final grafica loss y accuracy de las dos fases.
# Depende de: base_model, model, train_ds, val_ds, class_weights, callbacks

base_model.trainable = True
fine_tune_at = int(len(base_model.layers) * 0.8)
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False
for layer in base_model.layers[fine_tune_at:]:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks,
)

# el checkpoint conserva el mejor val_loss de ambas fases (es el mismo
# callback en los dos fit), se carga ese modelo
model = tf.keras.models.load_model(MODEL_PATH)
print(f"cargado el mejor modelo desde {MODEL_PATH}")

# curvas de entrenamiento de ambas fases
hist = {k: history_head.history[k] + history_fine.history[k] for k in history_head.history if k in history_fine.history}
fin_fase1 = len(history_head.history["loss"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metrica in zip(axes, ["loss", "accuracy"]):
    ax.plot(hist[metrica], label="train")
    ax.plot(hist[f"val_{metrica}"], label="val")
    ax.axvline(fin_fase1 - 0.5, color="gray", linestyle="--", label="inicio fine-tuning")
    ax.set_title(metrica)
    ax.set_xlabel("epoca")
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Celda 8: evaluacion en validacion
# Predice todo val, toma la clase con mayor probabilidad (argmax) y
# calcula precision, recall y F1 por clase. Despues dibuja la matriz de
# confusion normalizada por fila: cada fila es la clase real y muestra
# a que clases la asigna el modelo, asi se ve que clases confunde.
# Depende de: model, val_ds, val_df, FINAL_CLASSES

val_probs = model.predict(val_ds)
val_pred = val_probs.argmax(axis=1)
val_true = val_df["label_idx"].values

print(classification_report(val_true, val_pred, target_names=FINAL_CLASSES, zero_division=0))

cm = confusion_matrix(val_true, val_pred, normalize="true")
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(cm, display_labels=FINAL_CLASSES).plot(
    ax=ax, cmap="Blues", values_format=".2f", xticks_rotation=45, colorbar=False
)
ax.set_title("matriz de confusion (validacion, normalizada por clase real)")
plt.tight_layout()
plt.show()

In [ ]:
# Celda 9: analisis de errores
# Para cada clase real muestra hasta 5 imagenes de validacion que el
# modelo clasifico mal, con la clase predicha y su probabilidad.
# Sirve para detectar etiquetas mal puestas o clases que conviene juntar.
# Depende de: val_df, val_pred, val_probs, FINAL_CLASSES (celda 8)

errores_df = val_df.copy()
errores_df["pred_idx"] = val_pred
errores_df["prob_pred"] = val_probs.max(axis=1)
errores_df = errores_df[errores_df["pred_idx"] != errores_df["label_idx"]]

print(f"Errores en validacion: {len(errores_df)} de {len(val_df)}")

N_POR_CLASE = 5
for cls in FINAL_CLASSES:
    errores_cls = errores_df[errores_df["label"] == cls]
    if errores_cls.empty:
        print(f"{cls}: sin errores")
        continue
    muestra = errores_cls.sample(min(N_POR_CLASE, len(errores_cls)), random_state=42)
    fig, axes = plt.subplots(1, N_POR_CLASE, figsize=(4 * N_POR_CLASE, 4))
    for ax in axes:
        ax.axis("off")
    for ax, (_, fila) in zip(axes, muestra.iterrows()):
        ax.imshow(Image.open(fila["image_path"]))
        ax.set_title(f"pred: {FINAL_CLASSES[fila['pred_idx']]}\n({fila['prob_pred']:.2f})", fontsize=9)
    fig.suptitle(f"REAL: {cls} ({len(errores_cls)} errores)", fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# Celda 10: evaluacion final en test
# Se corre UNA sola vez, con el modelo ya decidido (sin volver a cambiar
# nada segun este resultado). Muestra accuracy, reporte por clase y
# matriz de confusion sobre test: este es el resultado que se reporta.
# Depende de: model, test_ds, test_df, FINAL_CLASSES

test_probs = model.predict(test_ds)
test_pred = test_probs.argmax(axis=1)
test_true = test_df["label_idx"].values

print(f"accuracy en test: {(test_pred == test_true).mean():.3f}\n")
print(classification_report(test_true, test_pred, target_names=FINAL_CLASSES, zero_division=0))

cm_test = confusion_matrix(test_true, test_pred, normalize="true")
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(cm_test, display_labels=FINAL_CLASSES).plot(
    ax=ax, cmap="Blues", values_format=".2f", xticks_rotation=45, colorbar=False
)
ax.set_title("matriz de confusion (test, normalizada por clase real)")
plt.tight_layout()
plt.show()

In [ ]:
# Celda 11: guardar modelo y configuracion
# El modelo ya quedo guardado por el checkpoint en
# ../models/modelo_clasificacion2.keras (el mejor de las dos fases), aca
# no se vuelve a guardar para no pisarlo. Se guardan al lado la lista de
# clases en el orden de las salidas del modelo y GRUPOS_FINAL (que clases
# originales forman cada superclase). Con esto se puede reproducir el entrenamiento y predecir
# imagenes nuevas. Incluye una funcion de prediccion de ejemplo.
# Depende de: model, MODEL_PATH (celda 6), FINAL_CLASSES, GRUPOS_FINAL, IMG_SIZE

with open("../models/modelo_clasificacion2_clases.json", "w") as f:
    json.dump(FINAL_CLASSES, f, indent=2, ensure_ascii=False)

with open("../models/modelo_clasificacion2_grupos.json", "w") as f:
    json.dump(GRUPOS_FINAL, f, indent=2, ensure_ascii=False)

print(f"modelo en {MODEL_PATH}, clases y grupos guardados en ../models/")

def predecir_dano(ruta_imagen):
    img = tf.io.read_file(ruta_imagen)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    probs = model.predict(tf.expand_dims(img, 0), verbose=0)[0]
    orden = probs.argsort()[::-1]
    return [(FINAL_CLASSES[i], float(probs[i])) for i in orden]

# predecir_dano("data_clasificacion/images/alguna_imagen.jpg")